# Tutorial 02: excited states – energies, NACs, dipoles and SOCs

*Author: Rhyan Barrett*

CH₂S in a droplet of 100 waters (SHARC 4's QM/MM example). The labels come from SHARC's LVC model of CH₂S,
which gives energies, gradients, NACs, dipoles and SOCs of S0, S1, T1, T2 in the field of the water charges.
Steps 1–2 need SHARC 4 (`$SHARC`). Details: [README](README.md).

In [ ]:
import os, subprocess, sys
import numpy as np
from IPython.display import Image, display

QUICK = os.environ.get("FIELDMACE_TUTORIAL_QUICK") == "1"   # tiny settings for automatic checks
os.environ.setdefault("OMP_NUM_THREADS", "4")

def run(cmd, env=None):
    """Run a tutorial step and show its output (stops the notebook if the step fails)."""
    print("$", cmd)
    p = subprocess.run(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                       env={**os.environ, **(env or {})})
    lines = [l for l in p.stdout.splitlines() if "Warning" not in l and "warnings.warn" not in l]
    print("\n".join(lines[-60:]))
    if p.returncode:
        raise RuntimeError(f"step failed: {cmd}")
assert os.environ.get("SHARC"), "set $SHARC (SHARC 4 bin directory) before starting Jupyter"

## Step 1: sampling

In [ ]:
run(f"python 01_sample_configurations.py --n-frames {80 if QUICK else 1000} --equil-ps {0.5 if QUICK else 20} --n-initconds {4 if QUICK else 10}")

## Step 2: LVC labels

4 spatial states (S0, S1, T1, T2). SOCs: the upper triangle of the 8×8 SOC matrix of all SHARC states with M_S,
as (re, im) pairs: 56 numbers per frame (`--soc_num=56`). The states' signs are aligned with the LVC
reference states, so the SOC vector has consistent phases; 10% gas-phase frames are added.

In [ ]:
run("python 02_label_with_lvc.py")

In [ ]:
import ase.io
frames = ase.io.read("data/train.xyz", index=":")
x = [f for f in frames if f.info.get("config_type") == "solvated"][0]
for k in ("REF_energy", "REF_forces", "REF_mm_forces", "REF_nacs", "REF_dipoles", "REF_socs"):
    print(f"{k:14s}", np.shape(x.info[k]))
E = np.array([np.ravel(f.info["REF_energy"]) for f in frames])
import matplotlib.pyplot as plt
plt.figure(figsize=(5, 3))
for s, n in enumerate(["S0", "S1", "T1", "T2"]):
    plt.hist(E[:, s] - E[:, 0].mean(), bins=40, alpha=0.6, label=n)
plt.xlabel("energy - <E(S0)> (eV)"); plt.legend(); plt.show()
soc = x.info["REF_socs"].reshape(-1, 2) @ np.array([1, 1j])
H = np.zeros((8, 8), complex); H[np.triu_indices(8, 1)] = soc
print("|SOC| matrix (cm-1), upper triangle:"); print(np.round(np.abs(H) * 8065.54, 1))

## Step 3: training

In [ ]:
env = {"EPOCHS": "2", "CHANNELS": "8"} if QUICK else {}
run("bash 03_train.sh", env=env)

## Step 4: evaluation (couplings compared with the best sign, as in the loss)

In [ ]:
run("python 04_evaluate.py")
display(Image("results/parity_excited_states.png"))

## Step 5: ASE and SHARC calculators

`MACECalculator` for all properties; `SharcCalculator` for SHARC's QM.out layout (what the SHARC interface
returns).

In [ ]:
run("python 05_calculators.py")
display(Image("results/scan_CS.png"))

In [ ]:
# the same by hand
from mace.calculators import MACECalculator
calc = MACECalculator("models/ch2s_es.model", device="cpu", state=1)
test = [f for f in ase.io.read("data/test.xyz", index=":") if "mm_charges" in f.info]
a = test[0].copy(); a.info = {"mm_positions": test[0].info["mm_positions"], "mm_charges": test[0].info["mm_charges"]}
calc.calculate(a)
print("energies (eV):", calc.results["energies"])
print("LVC        :", np.ravel(test[0].info["REF_energy"]))
print("S1 forces (eV/A):\n", calc.results["forces"])